# IMPORTS

In [15]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv("cardio_data_processed.csv")

# Preprocessing & Feature Engineering:

In [3]:
df.drop('id', axis=1, inplace = True) # Dropped id becoz its useless for prediction...
df["age"] = (df.age/365).astype(int) # Converting Age in years as its given in days...
df.drop('age_years', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category_encoded', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...

In [4]:
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
df["mean_arterial_pressure"] = df["ap_hi"] + 2*df["ap_lo"]/3

In [5]:
#Checking if there any missing or false value & dropping that...
df[['ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure']].describe()
print((df['pulse_pressure'] < 0).sum()) #Since i got 3 value whose pulse_pressure is < 0 which make no sense as there is an error, so i gonna drop this 3 rows
df = df[df['pulse_pressure'] >= 0] #It removed those 3rows but index still show 0 to 68204 so we need to reset index...
df = df.reset_index(drop=True) #index reset...

3


# Train Test Split:

In [6]:
X = df.drop('cardio', axis = 1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

# OPTUNA FOR LR:

In [8]:
def objective(trial):
    """
    Objective function for Optuna to optimize LR hyperparameters
    """
    
    # Define hyperparameter search space
    C = trial.suggest_float('C', 0.001, 100, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    
    # Match solver with penalty (liblinear only supports l1 or l2, saga supports both)
    if penalty == 'l1':
        solver = 'liblinear'
    else:  # penalty == 'l2'
        solver = trial.suggest_categorical('solver', ['liblinear', 'lbfgs', 'saga'])
    
    max_iter = trial.suggest_int('max_iter', 1000, 3000)
    
    try:
        # Create pipeline with suggested hyperparameters
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                random_state=10,
                C=C,
                penalty=penalty,
                solver=solver,
                max_iter=max_iter,
                n_jobs=-1,  # Use all CPU cores
                warm_start=False
            ))
        ])
        
        # Use Stratified 5-Fold CV to evaluate
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)
        
        # Define scoring metrics
        scoring = {
            'accuracy': 'accuracy',
            'precision': 'precision_weighted',
            'recall': 'recall_weighted',
            'f1': 'f1_weighted',
            'roc_auc': 'roc_auc_ovr_weighted'
        }
        
        # Perform cross-validation with timeout handling
        cv_results = cross_validate(
            pipeline, 
            X_train, 
            y_train, 
            cv=cv, 
            scoring=scoring,
            return_train_score=False,
            n_jobs=1  # Single job for CV
        )
        
        # Return mean Accuracy for optimization
        mean_accuracy = cv_results['test_accuracy'].mean()
        
        return mean_accuracy
        
    except Exception as e:
        # If convergence fails, return a very poor score
        print(f"Trial failed with error: {str(e)[:50]}")
        return 0.0


# Create study object with better pruning
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=10, n_startup_trials=20),
    pruner=optuna.pruners.PercentilePruner(percentile=25)
)

# Run optimization
print("Starting Optuna optimization for Logistic Regression...")
print("=" * 70)

study.optimize(
    objective, 
    n_trials=100, 
    show_progress_bar=True,
    gc_after_trial=True  # Force garbage collection after each trial
)

# Get best trial
best_trial_lr = study.best_trial

print("\n" + "=" * 70)
print("OPTUNA TUNING RESULTS - LOGISTIC REGRESSION")
print("=" * 70)
print(f"\nBest Accuracy: {best_trial_lr.value:.4f}")
print(f"Number of Completed Trials: {len([t for t in study.trials if t.state.name == 'COMPLETE'])}")
print(f"\nBest Hyperparameters:")
print("-" * 70)
for key, value in best_trial_lr.params.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")

print("=" * 70)

# Store best params for next steps
best_params_lr = best_trial_lr.params

[I 2026-05-06 01:28:44,024] A new study created in memory with name: no-name-0426fa99-94cc-4493-84ef-f0266246a0a1


Starting Optuna optimization for Logistic Regression...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-06 01:28:46,058] Trial 0 finished with value: 0.7255180704064914 and parameters: {'C': 7.187919693377491, 'penalty': 'l2', 'solver': 'liblinear', 'max_iter': 1396}. Best is trial 0 with value: 0.7255180704064914.
[I 2026-05-06 01:28:48,243] Trial 1 finished with value: 0.7255363988522393 and parameters: {'C': 6.348243270946395, 'penalty': 'l1', 'max_iter': 2371}. Best is trial 1 with value: 0.7255363988522393.
[I 2026-05-06 01:28:50,329] Trial 2 finished with value: 0.7255180704064914 and parameters: {'C': 58.474528815522426, 'penalty': 'l2', 'solver': 'liblinear', 'max_iter': 1584}. Best is trial 1 with value: 0.7255363988522393.
[I 2026-05-06 01:28:52,404] Trial 3 finished with value: 0.7255730540642291 and parameters: {'C': 38.803474329218346, 'penalty': 'l1', 'max_iter': 1284}. Best is trial 3 with value: 0.7255730540642291.
[I 2026-05-06 01:29:10,414] Trial 4 finished with value: 0.7253714478790262 and parameters: {'C': 0.0735705156954108, 'penalty': 'l1', 'max_iter': 1

# RFE Feature Selection:

In [17]:
print("Starting RFE Feature Selection for Logistic Regression...")
print("=" * 80)
print(f"Using best hyperparameters from Optuna:")
for key, value in best_params_lr.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")
print("=" * 80)

# Get solver - if not in best_params_lr, use default
solver_to_use = best_params_lr.get('solver', 'liblinear')

# Feature count range to test (8-14 features)
feature_counts = range(8, 15)  # 8, 9, 10, 11, 12, 13, 14

# Dictionary to store results
rfe_results_lr = {}

# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Loop through each feature count
for n_features in feature_counts:
    print(f"\nTesting with {n_features} features...")
    
    # Store metrics for all folds
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    fold_roc_aucs = []
    
    # CV Loop - RFE EMBEDDED INSIDE
    fold_idx = 0
    for train_idx, val_idx in cv.split(X_train, y_train):
        fold_idx += 1
        
        # Split data into train and validation
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Create pipeline with RFE
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("rfe", RFE(
                estimator=LogisticRegression(
                    random_state=10,
                    C=best_params_lr['C'],
                    penalty=best_params_lr['penalty'],
                    solver=solver_to_use,
                    max_iter=best_params_lr['max_iter'],
                    n_jobs=-1
                ),
                n_features_to_select=n_features,
                step=1
            )),
            ("model", LogisticRegression(
                random_state=10,
                C=best_params_lr['C'],
                penalty=best_params_lr['penalty'],
                solver=solver_to_use,
                max_iter=best_params_lr['max_iter'],
                n_jobs=-1
            ))
        ])
        
        # Fit on TRAINING fold (RFE learns feature importance from training fold only)
        pipeline.fit(X_fold_train, y_fold_train)
        
        # Predict on VALIDATION fold
        y_pred = pipeline.predict(X_fold_val)
        y_pred_proba = pipeline.predict_proba(X_fold_val)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_fold_val, y_pred)
        precision = precision_score(y_fold_val, y_pred, average='weighted')
        recall = recall_score(y_fold_val, y_pred, average='weighted')
        f1 = f1_score(y_fold_val, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
        
        # Store results
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)
        fold_roc_aucs.append(roc_auc)
    
    # Store mean and std for this feature count
    rfe_results_lr[n_features] = {
        'accuracy': np.array(fold_accuracies),
        'precision': np.array(fold_precisions),
        'recall': np.array(fold_recalls),
        'f1': np.array(fold_f1s),
        'roc_auc': np.array(fold_roc_aucs)
    }
    
    # Display results for this feature count
    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)
    print(f"  Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")

# ============================================================
# SELECT BEST FEATURE COUNT (based on Accuracy)
# ============================================================

print("\n" + "=" * 80)
print("RFE RESULTS SUMMARY (Logistic Regression)")
print("=" * 80)

# Create results dataframe
results_data = {}
for n_features in feature_counts:
    results_data[n_features] = {
        'Accuracy': f"{rfe_results_lr[n_features]['accuracy'].mean():.3f} ± {rfe_results_lr[n_features]['accuracy'].std():.3f}",
        'Precision': f"{rfe_results_lr[n_features]['precision'].mean():.3f} ± {rfe_results_lr[n_features]['precision'].std():.3f}",
        'Recall': f"{rfe_results_lr[n_features]['recall'].mean():.3f} ± {rfe_results_lr[n_features]['recall'].std():.3f}",
        'F1 Score': f"{rfe_results_lr[n_features]['f1'].mean():.3f} ± {rfe_results_lr[n_features]['f1'].std():.3f}",
        'ROC AUC': f"{rfe_results_lr[n_features]['roc_auc'].mean():.3f} ± {rfe_results_lr[n_features]['roc_auc'].std():.3f}"
    }

results_df = pd.DataFrame(results_data).T
print("\n")
print(results_df)

# Find best feature count based on Accuracy
best_feature_count_lr = max(
    rfe_results_lr.keys(),
    key=lambda x: rfe_results_lr[x]['accuracy'].mean()
)

best_accuracy = rfe_results_lr[best_feature_count_lr]['accuracy'].mean()
best_accuracy_std = rfe_results_lr[best_feature_count_lr]['accuracy'].std()

print("\n" + "=" * 80)
print("BEST FEATURE COUNT")
print("=" * 80)
print(f"\nBest Number of Features: {best_feature_count_lr}")
print(f"Best Accuracy: {best_accuracy:.3f} ± {best_accuracy_std:.3f}")
print("\nAll Metrics for Best Feature Count:")
print("-" * 80)
for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    metric_values = rfe_results_lr[best_feature_count_lr][metric_name]
    mean = metric_values.mean()
    std = metric_values.std()
    print(f"  {metric_name.upper():10s}: {mean:.3f} ± {std:.3f}")

print("=" * 80)

# Store best feature count for next step
best_rfe_features_lr = best_feature_count_lr
print(f"\n✓ Best feature count saved: {best_rfe_features_lr}")

Starting RFE Feature Selection for Logistic Regression...
Using best hyperparameters from Optuna:
  C: 38.803474
  penalty: l1
  max_iter: 1284

Testing with 8 features...
  Accuracy: 0.7244 ± 0.0051

Testing with 9 features...
  Accuracy: 0.7249 ± 0.0052

Testing with 10 features...
  Accuracy: 0.7251 ± 0.0050

Testing with 11 features...
  Accuracy: 0.7251 ± 0.0050

Testing with 12 features...
  Accuracy: 0.7254 ± 0.0046

Testing with 13 features...
  Accuracy: 0.7255 ± 0.0047

Testing with 14 features...
  Accuracy: 0.7256 ± 0.0045

RFE RESULTS SUMMARY (Logistic Regression)


         Accuracy      Precision         Recall       F1 Score        ROC AUC
8   0.724 ± 0.005  0.727 ± 0.005  0.724 ± 0.005  0.723 ± 0.005  0.788 ± 0.007
9   0.725 ± 0.005  0.728 ± 0.005  0.725 ± 0.005  0.724 ± 0.005  0.789 ± 0.007
10  0.725 ± 0.005  0.728 ± 0.005  0.725 ± 0.005  0.724 ± 0.005  0.789 ± 0.007
11  0.725 ± 0.005  0.728 ± 0.005  0.725 ± 0.005  0.724 ± 0.005  0.789 ± 0.007
12  0.725 ± 0.005  0.728

# Final LR Pipeline with Results:

In [19]:
# Get solver
solver_to_use = best_params_lr.get('solver', 'liblinear')

# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Store metrics for all folds
fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []
fold_roc_aucs = []

print("\nRunning 5-Fold Stratified Cross-Validation...")
print("-" * 80)
print(f"{'Fold':<6} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'ROC AUC':<12}")
print("-" * 80)

# CV Loop - RFE EMBEDDED INSIDE
fold_idx = 0
for train_idx, val_idx in cv.split(X_train, y_train):
    fold_idx += 1
    
    # Split data into train and validation
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create pipeline with RFE and best feature count
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("rfe", RFE(
            estimator=LogisticRegression(
                random_state=10,
                C=best_params_lr['C'],
                penalty=best_params_lr['penalty'],
                solver=solver_to_use,
                max_iter=best_params_lr['max_iter'],
                n_jobs=-1
            ),
            n_features_to_select=best_rfe_features_lr,
            step=1
        )),
        ("model", LogisticRegression(
            random_state=10,
            C=best_params_lr['C'],
            penalty=best_params_lr['penalty'],
            solver=solver_to_use,
            max_iter=best_params_lr['max_iter'],
            n_jobs=-1
        ))
    ])
    
    # Fit on TRAINING fold
    pipeline.fit(X_fold_train, y_fold_train)
    
    # Predict on VALIDATION fold
    y_pred = pipeline.predict(X_fold_val)
    y_pred_proba = pipeline.predict_proba(X_fold_val)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_fold_val, y_pred)
    precision = precision_score(y_fold_val, y_pred, average='weighted')
    recall = recall_score(y_fold_val, y_pred, average='weighted')
    f1 = f1_score(y_fold_val, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
    
    # Store results
    fold_accuracies.append(accuracy)
    fold_precisions.append(precision)
    fold_recalls.append(recall)
    fold_f1s.append(f1)
    fold_roc_aucs.append(roc_auc)
    
    # Print fold results
    print(f"{fold_idx:<6} {accuracy:<12.4f} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {roc_auc:<12.4f}")

# Calculate and display summary
print("\n" + "=" * 80)
print("FINAL PIPELINE RESULTS (Mean ± Std)")
print("=" * 80)

print("\n")
print(f"{'Accuracy':<12s}: {np.mean(fold_accuracies):.3f} ± {np.std(fold_accuracies):.3f}")
print(f"{'Precision':<12s}: {np.mean(fold_precisions):.3f} ± {np.std(fold_precisions):.3f}")
print(f"{'Recall':<12s}: {np.mean(fold_recalls):.3f} ± {np.std(fold_recalls):.3f}")
print(f"{'F1 Score':<12s}: {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"{'ROC AUC':<12s}: {np.mean(fold_roc_aucs):.3f} ± {np.std(fold_roc_aucs):.3f}")

print("\n" + "=" * 80)

# ============================================================
# TRAIN FINAL MODEL ON FULL TRAIN DATA
# ============================================================

final_pipeline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("rfe", RFE(
        estimator=LogisticRegression(
            random_state=10,
            C=best_params_lr['C'],
            penalty=best_params_lr['penalty'],
            solver=solver_to_use,
            max_iter=best_params_lr['max_iter'],
            n_jobs=-1
        ),
        n_features_to_select=best_rfe_features_lr,
        step=1
    )),
    ("model", LogisticRegression(
        random_state=10,
        C=best_params_lr['C'],
        penalty=best_params_lr['penalty'],
        solver=solver_to_use,
        max_iter=best_params_lr['max_iter'],
        n_jobs=-1
    ))
])

print("Training Final Model on full train data...")
final_pipeline_lr.fit(X_train, y_train)
print("✓ Final model trained on full training data!")
print("=" * 80)


Running 5-Fold Stratified Cross-Validation...
--------------------------------------------------------------------------------
Fold   Accuracy     Precision    Recall       F1 Score     ROC AUC     
--------------------------------------------------------------------------------
1      0.7342       0.7366       0.7342       0.7331       0.8016      
2      0.7249       0.7279       0.7249       0.7235       0.7876      
3      0.7245       0.7263       0.7245       0.7236       0.7893      
4      0.7210       0.7235       0.7210       0.7199       0.7808      
5      0.7232       0.7263       0.7232       0.7218       0.7857      

FINAL PIPELINE RESULTS (Mean ± Std)


Accuracy    : 0.726 ± 0.005
Precision   : 0.728 ± 0.004
Recall      : 0.726 ± 0.005
F1 Score    : 0.724 ± 0.005
ROC AUC     : 0.789 ± 0.007

Training Final Model on full train data...
✓ Final model trained on full training data!
